<a href="https://colab.research.google.com/github/ghizlane89/0__GenIA/blob/Bootcamp/W13_D2_DC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# %% Colab Cell 1: Setup + files
import os, json, textwrap, pathlib, subprocess, sys

BASE = "/content/agentic_rag_app"
os.makedirs(BASE, exist_ok=True)

# requirements.txt
open(f"{BASE}/requirements.txt","w").write("""streamlit
langchain>=0.2.12
langchain-community>=0.2.10
langchain-groq>=0.1.5
tavily-python>=0.4.0
faiss-cpu
tiktoken
python-dotenv
sentence-transformers
numpy
pydantic>=2.0.0
pyngrok==5.2.1
requests
""")

# .env.example
open(f"{BASE}/.env.example","w").write("""# --- Required (choose one path) ---
# If you use Groq directly:
GROQ_API_KEY=''

# If you use Cloudflare Workers AI instead of Groq:
CLOUDFLARE_ACCOUNT_ID=''
CLOUDFLARE_API_TOKEN=''
CF_MODEL=@cf/meta/llama-3.1-8b-instruct  # default model if using Cloudflare

# Web search (optional but recommended if you want retrieval over the web)
TAVILY_API_KEY=''

# --- Optional ---
GOOGLE_API_KEY=''
LANGCHAIN_API_KEY=''

# LangSmith / tracing (optional)
LANGCHAIN_TRACING_V2=true
LANGCHAIN_ENDPOINT=https://api.smith.langchain.com
LANGCHAIN_PROJECT=agentic-rag-daily-challenge
LANGCHAIN_CALLBACKS_BACKGROUND=true

# Model overrides
GROQ_MODEL=llama-3.1-8b-instant
""")

# rag_agent.py
open(f"{BASE}/rag_agent.py","w").write(r'''from __future__ import annotations
import os, glob, json
from typing import List, Dict, Any
from dataclasses import dataclass

import requests
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import TextLoader
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_groq import ChatGroq
from langchain.schema import Document
from langchain.prompts import ChatPromptTemplate

DEFAULT_GROQ_MODEL = os.getenv("GROQ_MODEL", "llama-3.1-8b-instant")
DEFAULT_CF_MODEL = os.getenv("CF_MODEL", "@cf/meta/llama-3.1-8b-instruct")

@dataclass
class RAGResponse:
    answer: str
    sources: List[Dict[str, str]]
    used_web: bool = False
    error: str | None = None

def _load_or_create_docs():
    data_dir = os.path.join(os.getcwd(), "data")
    paths = glob.glob(os.path.join(data_dir, "*.txt"))
    docs = []
    if paths:
        for p in paths:
            try:
                docs.extend(TextLoader(p, encoding="utf-8").load())
            except Exception:
                pass
    if not docs:
        docs = [
            Document(page_content="Agentic RAG = reason -> retrieve -> read -> synthesize, with citations.",
                     metadata={"source": "intro.txt"}),
            Document(page_content="FAISS supports fast similarity search; LangChain provides a retriever over it.",
                     metadata={"source": "faiss.txt"}),
        ]
    return docs

def _build_retriever(docs):
    splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=80)
    chunks = splitter.split_documents(docs)
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vs = FAISS.from_documents(chunks, embeddings)
    return vs.as_retriever(search_kwargs={"k": 4})

class _CFChat:
    """Minimal Cloudflare Workers AI client with an .invoke(messages) API compatible-ish with LangChain calls."""
    def __init__(self, account_id: str, api_token: str, model: str = DEFAULT_CF_MODEL, timeout: int = 45):
        self.account_id = account_id
        self.api_token = api_token
        self.model = model
        self.timeout = timeout

    def _endpoint(self) -> str:
        # https://api.cloudflare.com/client/v4/accounts/{account_id}/ai/run/{model}
        return f"https://api.cloudflare.com/client/v4/accounts/{self.account_id}/ai/run/{self.model}"

    def invoke(self, messages: List[Any]):
        # Concatenate message contents (system + human, etc.)
        contents = []
        for m in messages:
            contents.append(getattr(m, "content", str(m)))
        input_text = "\n\n".join(contents).strip()

        try:
            r = requests.post(
                self._endpoint(),
                headers={
                    "Authorization": f"Bearer {self.api_token}",
                    "Content-Type": "application/json",
                },
                json={"input_text": input_text},
                timeout=self.timeout,
            )
            r.raise_for_status()
            data = r.json()
            # Cloudflare returns {"success": true, "result": {"response": "...", ...}, ...}
            resp = (data.get("result") or {}).get("response") or ""
            if not resp:
                resp = json.dumps(data)[:2000]
            return type("Msg", (), {"content": resp})
        except Exception as e:
            return type("Msg", (), {"content": f"[CF ERROR] {e}"})

def _llm_or_simulation():
    # Priority 1: Groq if key present
    if os.getenv("GROQ_API_KEY"):
        return ChatGroq(temperature=0.2, model_name=DEFAULT_GROQ_MODEL, timeout=30)
    # Priority 2: Cloudflare Workers AI if creds present
    cf_id = os.getenv("CLOUDFLARE_ACCOUNT_ID")
    cf_tok = os.getenv("CLOUDFLARE_API_TOKEN")
    if cf_id and cf_tok:
        return _CFChat(account_id=cf_id, api_token=cf_tok, model=DEFAULT_CF_MODEL, timeout=45)
    # Fallback: simulation
    class _Sim:
        def invoke(self, messages):
            last = messages[-1]
            content = getattr(last, "content", str(last))
            return type("Msg", (), {"content": f"[SIMULATED ANSWER]\\n{content[:800]}"})
    return _Sim()

def answer_question(query: str, k: int = 4, use_web: bool = True) -> RAGResponse:
    try:
        docs = _load_or_create_docs()
        retriever = _build_retriever(docs)
        retrieved = retriever.get_relevant_documents(query)[:k]
        src_summaries = []
        for d in retrieved:
            m = d.metadata or {}
            src_summaries.append({
                "title": m.get("title") or m.get("source") or "local-doc",
                "url": m.get("url", ""),
                "source": m.get("source", ""),
                "snippet": d.page_content[:200]
            })

        web_results = []
        if use_web and os.getenv("TAVILY_API_KEY"):
            try:
                tavily = TavilySearchResults(max_results=5)
                web_results = tavily.invoke({"query": query}) or []
                for r in web_results:
                    src_summaries.append({
                        "title": r.get("title","web"),
                        "url": r.get("url",""),
                        "source": "tavily",
                        "snippet": (r.get("content") or "")[:200]
                    })
            except Exception:
                pass

        system = ("You are a helpful research agent. Answer ONLY from CONTEXT. "
                  "Use inline citations like [1],[2]. If unsure, say you don't know.")
        numbered = []
        for i, s in enumerate(src_summaries, start=1):
            tag = s.get("url") or s.get("source") or s.get("title") or f"doc{i}"
            numbered.append(f"[{i}] {tag} :: {s.get('snippet','')}")

        prompt = ChatPromptTemplate.from_messages([
            ("system", system),
            ("human", "QUESTION:\\n{q}\\n\\nCONTEXT:\\n{ctx}\\n\\nAnswer concisely with citations.")
        ])
        llm = _llm_or_simulation()
        msg = prompt.format_messages(q=query, ctx="\n".join(numbered) if numbered else "(no context)")
        out = llm.invoke(msg)
        return RAGResponse(
            answer=getattr(out, "content", str(out)),
            sources=[{"title": s["title"], "url": s.get("url",""), "source": s.get("source","")} for s in src_summaries],
            used_web=bool(web_results),
            error=None
        )
    except Exception as e:
        return RAGResponse(
            answer="Désolé, une erreur est survenue pendant le raisonnement.",
            sources=[], used_web=False, error=str(e)
        )
''')

# app.py
open(f"{BASE}/app.py","w").write(r'''import os, json
from pathlib import Path
import streamlit as st
from dotenv import load_dotenv

# Load .env and set LangSmith flags
load_dotenv(override=True)
os.environ.setdefault("LANGCHAIN_TRACING_V2", os.getenv("LANGCHAIN_TRACING_V2","false"))
os.environ.setdefault("LANGCHAIN_ENDPOINT", os.getenv("LANGCHAIN_ENDPOINT","https://api.smith.langchain.com"))
os.environ.setdefault("LANGCHAIN_PROJECT", os.getenv("LANGCHAIN_PROJECT","agentic-rag-daily-challenge"))
os.environ.setdefault("LANGCHAIN_CALLBACKS_BACKGROUND", os.getenv("LANGCHAIN_CALLBACKS_BACKGROUND","true"))

KEYS = {
    "GROQ_API_KEY": bool(os.getenv("GROQ_API_KEY")),
    "CLOUDFLARE_ACCOUNT_ID": bool(os.getenv("CLOUDFLARE_ACCOUNT_ID")),
    "CLOUDFLARE_API_TOKEN": bool(os.getenv("CLOUDFLARE_API_TOKEN")),
    "TAVILY_API_KEY": bool(os.getenv("TAVILY_API_KEY")),
    "GOOGLE_API_KEY": bool(os.getenv("GOOGLE_API_KEY")),
    "LANGCHAIN_API_KEY": bool(os.getenv("LANGCHAIN_API_KEY")),
}
MODELS = {
    "GROQ_MODEL": os.getenv("GROQ_MODEL", "llama-3.1-8b-instant"),
    "CF_MODEL": os.getenv("CF_MODEL", "@cf/meta/llama-3.1-8b-instruct"),
}

st.set_page_config(page_title="Agentic RAG • Daily Challenge", page_icon="🧠", layout="centered")
st.title("🧠 Agentic RAG + Tool-Using Agent (Daily Challenge) — Colab")

with st.expander("Environment status (keys present?)"):
    st.json({k: ("✅" if v else "❌") for k, v in KEYS.items()})
    st.caption(f"Models → GROQ_MODEL={MODELS['GROQ_MODEL']} • CF_MODEL={MODELS['CF_MODEL']}")

st.write("Tape une question puis **Submit**. Si aucune clé Groq ni Cloudflare n'est configurée, la réponse est simulée.")

query = st.text_input("Ta question")
use_web = st.toggle("Autoriser la recherche web (Tavily)", value=True)
submit = st.button("Submit", type="primary")

with st.expander("Voir agentic_rag.ipynb (brut)"):
    nb_path = Path("agentic_rag.ipynb")
    if nb_path.exists():
        try:
            raw = nb_path.read_text(encoding="utf-8")
            st.code(json.dumps(json.loads(raw), indent=2)[:100000])
        except Exception:
            st.code(nb_path.read_text(encoding="utf-8")[:100000])
    else:
        st.info("Fichier agentic_rag.ipynb introuvable.")

if submit and query.strip():
    try:
        from rag_agent import answer_question
        with st.spinner("Raisonner → récupérer → lire → synthétiser..."):
            resp = answer_question(query, use_web=use_web)
        if resp.error:
            st.error(f"Erreur : {resp.error}")
        st.subheader("Réponse")
        st.write(resp.answer)
        if resp.sources:
            st.subheader("Sources")
            for i, s in enumerate(resp.sources, start=1):
                label = s.get("title") or s.get("source") or f"source {i}"
                url = s.get("url")
                if url:
                    st.markdown(f"[{i}] **{label}** — {url}")
                else:
                    st.markdown(f"[{i}] **{label}**")
        st.caption("🔁 Web utilisé: " + ("Oui" if resp.used_web else "Non"))
    except Exception as e:
        st.warning("Impossible d'appeler l'agent, réponse simulée.")
        st.write(f"[SIMULATED ANSWER]\\n{e}")
''')

# minimal agentic_rag.ipynb (for display in app)
nb = {
 "cells":[
  {"cell_type":"markdown","metadata":{},"source":[
   "# Agentic RAG Notebook (Daily Challenge)\n",
   "Notebook vitrine (le code exécutable est dans `rag_agent.py`).\n",
   "- Retriever FAISS + embeddings sentence-transformers\n",
   "- Outil web Tavily (optionnel)\n",
   "- Groq **ou** Cloudflare Workers AI (ou simulation sans clé)\n",
   "- Boucle raisonner → récupérer → lire → synthétiser avec citations\n"]},
 ],
 "metadata":{"kernelspec":{"display_name":"Python 3","language":"python","name":"python3"}},
 "nbformat":4,"nbformat_minor":5
}
open(f"{BASE}/agentic_rag.ipynb","w").write(json.dumps(nb, ensure_ascii=False))

print("✅ Projet créé dans", BASE)




✅ Projet créé dans /content/agentic_rag_app


In [ ]:
# %% Colab Cell 2: Install deps
!pip -q install -r /content/agentic_rag_app/requirements.txt


In [ ]:
# %% Colab Cell 3: Set secrets (edit then run)
from dotenv import set_key
import os

env_path = "/content/agentic_rag_app/.env"

# ==== Choisis un des deux chemins ====

# --- OPTION 1: GROQ ---
GROQ_API_KEY   = ""   # colle ta clé Groq ici (si tu passes par Groq)

# --- OPTION 2: CLOUDFLARE Workers AI ---
CLOUDFLARE_ACCOUNT_ID = ""   # colle ton Cloudflare Account ID ici
CLOUDFLARE_API_TOKEN  = ""   # colle ton Cloudflare API Token ici
CF_MODEL = "@cf/meta/llama-3.1-8b-instruct"

# --- Web search (optionnel mais recommandé) ---
TAVILY_API_KEY = ""   # colle ta clé Tavily ici

# --- Autres clés optionnelles ---
GOOGLE_API_KEY     = ""
LANGCHAIN_API_KEY  = ""
LANGCHAIN_TRACING_V2 = "true"
LANGCHAIN_ENDPOINT   = "https://api.smith.langchain.com"
LANGCHAIN_PROJECT    = "agentic-rag-daily-challenge"
LANGCHAIN_CALLBACKS_BACKGROUND = "true"

kv = {
    "GROQ_API_KEY": GROQ_API_KEY,
    "CLOUDFLARE_ACCOUNT_ID": CLOUDFLARE_ACCOUNT_ID,
    "CLOUDFLARE_API_TOKEN": CLOUDFLARE_API_TOKEN,
    "CF_MODEL": CF_MODEL,
    "TAVILY_API_KEY": TAVILY_API_KEY,
    "GOOGLE_API_KEY": GOOGLE_API_KEY,
    "LANGCHAIN_API_KEY": LANGCHAIN_API_KEY,
    "LANGCHAIN_TRACING_V2": LANGCHAIN_TRACING_V2,
    "LANGCHAIN_ENDPOINT": LANGCHAIN_ENDPOINT,
    "LANGCHAIN_PROJECT": LANGCHAIN_PROJECT,
    "LANGCHAIN_CALLBACKS_BACKGROUND": LANGCHAIN_CALLBACKS_BACKGROUND,
}

# Création du fichier .env si manquant
os.makedirs(os.path.dirname(env_path), exist_ok=True)
open(env_path,"a").close()

# Écriture dans le fichier .env
for k,v in kv.items():
    if v is not None:
        set_key(env_path, k, v)

# Charger dans l'environnement courant
for k,v in kv.items():
    if v:
        os.environ[k] = v

print("✅ Clés écrites dans .env et chargées en mémoire.")



✅ Clés écrites dans .env et chargées en mémoire.


In [ ]:
# %% Colab Cell 4B (fix): Run Streamlit with Cloudflared (no account needed)
import os, time, subprocess, re

APP_DIR = "/content/agentic_rag_app"
PORT = 8501

# Télécharger cloudflared binaire si absent
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

# Lancer streamlit en arrière-plan
env = os.environ.copy()
streamlit_proc = subprocess.Popen(
    ["streamlit", "run", "app.py", "--server.port", str(PORT), "--server.headless", "true",
     "--browser.gatherUsageStats", "false"],
    cwd=APP_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

time.sleep(3)

# Lancer cloudflared tunnel
cloudflared_proc = subprocess.Popen(
    ["/usr/local/bin/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}", "--no-autoupdate"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

print("⏳ Démarrage du tunnel Cloudflared...")
public_url = None
for line in cloudflared_proc.stdout:
    if "trycloudflare.com" in line:
        m = re.search(r"https?://[0-9a-zA-Z\-\.]+trycloudflare\.com", line)
        if m:
            public_url = m.group(0)
            print("🌐 Public URL:", public_url)
            print("🚀 Streamlit tourne. Ouvre l’URL et laisse cette cellule tourner.")
            break
    if "ERR" in line or "error" in line.lower():
        print("⚠️", line.strip())

# Petit extrait des logs Streamlit
time.sleep(2)
print("\n🗒️ Logs Streamlit:")
for _ in range(15):
    l = streamlit_proc.stdout.readline()
    if not l: break
    print(l.rstrip())




⏳ Démarrage du tunnel Cloudflared...
🌐 Public URL: https://folder-small-wants-engineer.trycloudflare.com
🚀 Streamlit tourne. Ouvre l’URL et laisse cette cellule tourner.

🗒️ Logs Streamlit:
2025-08-20 14:10:22.879 Port 8501 is already in use
